# Notebook 1 — Read & join the tables

**Job of this notebook:** read every Olist table, understand each one on its
own, then build a single ML table with **one row per order**.

**Reads:** the local Olist database (no upstream notebook artifact).
**Writes:** `data/interim/ml_table.parquet` — one row per order.

Rule of thumb for this notebook: keep the analysis light. We are not doing
EDA here — that's Notebook 4. We only look at each table long enough to
join it correctly: row counts, keys, duplicates, and grain (what does one
row mean?).


In [ ]:
import sys
sys.path.append("../src")

import pandas as pd
from config import get_engine, ML_TABLE_PATH

pd.set_option("display.max_columns", 50)
engine = get_engine()


## 1. Read every table and look at it on its own

In [ ]:
TABLES = [
    "olist_orders_dataset",
    "olist_order_items_dataset",
    "olist_order_payments_dataset",
    "olist_order_reviews_dataset",
    "olist_customers_dataset",
    "olist_sellers_dataset",
    "olist_products_dataset",
    "olist_geolocation_dataset",
    "product_category_name_translation",
]

raw = {name: pd.read_sql_table(name, engine) for name in TABLES}

for name, df in raw.items():
    print(f"{name:40s} rows={len(df):8,d}  cols={df.shape[1]:3d}")


## 2. Check keys, duplicates, and grain of each table

This is the "what is one row here?" pass. It decides how each table needs
to be aggregated before it can be joined onto orders.


In [ ]:
orders = raw["olist_orders_dataset"]
items = raw["olist_order_items_dataset"]
payments = raw["olist_order_payments_dataset"]
reviews = raw["olist_order_reviews_dataset"]
customers = raw["olist_customers_dataset"]
sellers = raw["olist_sellers_dataset"]
products = raw["olist_products_dataset"]
geo = raw["olist_geolocation_dataset"]
cat_translation = raw["product_category_name_translation"]

print("orders: one row per order?      ", orders["order_id"].is_unique)
print("customers: one row per customer?", customers["customer_id"].is_unique)
print("products: one row per product?  ", products["product_id"].is_unique)
print("sellers: one row per seller?    ", sellers["seller_id"].is_unique)

# These two are NOT one row per order — that's the whole point of the
# "aggregate before you join" warning in the task sheet.
print()
print("order_items rows per order (should often be >1):")
print(items.groupby("order_id").size().describe())

print()
print("order_payments rows per order (multiple payment installments/methods):")
print(payments.groupby("order_id").size().describe())


In [ ]:
# Duplicate rows, exact duplicates
for name, df in raw.items():
    dupes = df.duplicated().sum()
    if dupes:
        print(f"{name}: {dupes} fully duplicated rows")


## 3. Aggregate order_items and order_payments to one row per order

`order_items` has one row per item, and `order_payments` has one row per
payment installment/method. Both need to be collapsed to order grain
*before* joining, or the join will fan out and silently multiply rows.


In [ ]:
items_agg = (
    items.groupby("order_id")
    .agg(
        n_items=("order_item_id", "count"),
        n_distinct_products=("product_id", "nunique"),
        n_distinct_sellers=("seller_id", "nunique"),
        total_price=("price", "sum"),
        total_freight_value=("freight_value", "sum"),
        avg_item_price=("price", "mean"),
        max_shipping_limit_date=("shipping_limit_date", "max"),
        # keep the first product/seller too, for product/seller-level features later
        product_id=("product_id", "first"),
        seller_id=("seller_id", "first"),
    )
    .reset_index()
)

payments_agg = (
    payments.groupby("order_id")
    .agg(
        n_payment_installments_rows=("payment_sequential", "count"),
        total_payment_value=("payment_value", "sum"),
        max_installments=("payment_installments", "max"),
        # most common payment type for the order
        payment_type=("payment_type", lambda s: s.mode().iat[0] if not s.mode().empty else None),
    )
    .reset_index()
)

reviews_agg = (
    reviews.sort_values("review_creation_date")
    .groupby("order_id")
    .agg(review_score=("review_score", "last"))
    .reset_index()
)

print(items_agg.shape, payments_agg.shape, reviews_agg.shape)
items_agg.head()


## 4. Bring in product, seller, customer dimension info

In [ ]:
products_enriched = products.merge(cat_translation, on="product_category_name", how="left")

products_small = products_enriched[[
    "product_id", "product_category_name_english",
    "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm",
]]

sellers_small = sellers[["seller_id", "seller_zip_code_prefix", "seller_city", "seller_state"]]
customers_small = customers[[
    "customer_id", "customer_unique_id",
    "customer_zip_code_prefix", "customer_city", "customer_state",
]]


## 5. Join everything onto orders — one row per order

`orders` is the spine (already one row per order). Everything else is
aggregated to order or product/seller/customer grain before joining, so
this merge chain can't fan rows out.


In [ ]:
ml_table = (
    orders
    .merge(items_agg, on="order_id", how="left")
    .merge(payments_agg, on="order_id", how="left")
    .merge(reviews_agg, on="order_id", how="left")
    .merge(customers_small, on="customer_id", how="left")
    .merge(products_small, on="product_id", how="left")
    .merge(sellers_small, on="seller_id", how="left")
)

print("ml_table shape:", ml_table.shape)
assert ml_table["order_id"].is_unique, "join fanned out — one row per order was violated"
ml_table.head()


## 6. Sanity checks before saving

Quick check that we didn't lose or duplicate orders in the join chain.


In [ ]:
assert len(ml_table) == len(orders), "row count changed vs. the orders spine table"
print("Missing after join (top 15):")
print(ml_table.isna().sum().sort_values(ascending=False).head(15))


## Artifact: `ml_table.parquet`

One row per order. This is what Notebook 2 reads to build the label.


In [ ]:
ml_table.to_parquet(ML_TABLE_PATH, index=False)
print(f"Saved {ml_table.shape} to {ML_TABLE_PATH}")
